# M06C: Capstone #3 — Multi-Tool Agent

Build an AI agent that connects to a live weather API and a customer database using function calling.

**Topics:**
- REST API integration (Open-Meteo)
- SQLite database queries and updates
- Multi-tool orchestration with error handling

---

## 🔧 Step 1: Setup

In [ ]:
import os
import json
import requests
import sqlite3
from pathlib import Path
from dotenv import load_dotenv

import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

INSTRUCTIONS = (
    "You are a helpful assistant. "
    "Use the supplied tools to answer user questions."
)


print(f"✅ Setup complete: Using {MODEL}!")

**Note:** The `instructions` parameter is optional. We're using it to keep the model focused, but function calling works without it.

---

## 🌤️ Part 1: Weather API Integration

Open-Meteo provides live weather data (no key required). We use real APIs to access up-to-date external data.

In [ ]:
def get_coordinates(city):
    """Get lat/lon from Open-Meteo geocoding API."""
    try:
        # Open-Meteo: free weather API, no key required
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=5
        )
        response.raise_for_status()    # Raises exception for 4xx/5xx status codes
        results = response.json().get("results", [])
        if results:
            return {"lat": results[0]["latitude"], "lon": results[0]["longitude"]}
        return None
    except Exception:
        return None


# --------------------------------------------------------------
print("✅ Geocoding function ready")

### Test the Geocoding Function

In [ ]:
# Test it
print("Testing geocoding API...")
print(get_coordinates("San Francisco"))
print(get_coordinates("Tokyo"))

---

### Define the Weather Function

In [ ]:
def get_live_weather(location):
    """Get real weather data from API."""
    try:
        location = location.strip()
        
        # Look up coordinates for this city
        coords = get_coordinates(location)
        if not coords:
            return {"error": "City not found"}
        
        # Field names: temperature_2m = temp at 2 meters, wind_speed_10m = wind at 10 meters
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "current": "temperature_2m,wind_speed_10m,weather_code"
        }
        
        # Make the live API call (5 second timeout)
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()  # Raises exception for 4xx/5xx status codes
        
        # Parse the JSON response
        data = response.json()
        current = data.get("current", {})
        
        return {
            "location": location,
            "temperature_c": current.get("temperature_2m"),
            "windspeed_kmh": current.get("wind_speed_10m"),
            "weather_code": current.get("weather_code")
        }
    
    # Specific error handling for different failure modes
    except requests.Timeout:
        return {"error": "API timeout - try again"}
    except requests.RequestException as e:
        return {"error": f"API error: {str(e)}"}
    except Exception as e:
        return {"error": f"Error: {str(e)}"}


# --------------------------------------------------------------
print("✅ Weather function ready")

### Test the Weather Function

In [ ]:
# Test it
print("Testing weather API...")
print(get_live_weather("Tokyo"))
print(get_live_weather("xyz not a city"))  # Tests error handling

---

### Define the Weather Tool

In [ ]:
# Weather tool schema (Responses API format)
weather_tools = [
    {
        "type": "function",
        "name": "get_live_weather",
        "description": "Get current weather for a city using live API data",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g. San Francisco"
                }
            },
            "required": ["location"]
        }
    }
]

weather_functions = {"get_live_weather": get_live_weather}


# --------------------------------------------------------------
print("✅ Weather API integration ready")

### Test with run_conversation

In [ ]:
def run_conversation(user_message, tools, available_functions, max_turns=5):
    """Run a function-calling loop. Stops when model returns no tool calls."""
    
    # Each item must be a dict when using a list
    conversation_items = [{"role": "user", "content": user_message}]
    
    # Loop until no tool calls (max turns to prevent infinite loops)
    for i, _ in enumerate(range(max_turns)):
        response = client.responses.create(
            model=MODEL,
            input=conversation_items,
            tools=tools
        )
        
        # Filter for function calls only
        tool_items = [
            item for item in response.output
            if item.type == "function_call"
        ]
        
        # No tool calls — return final response
        if not tool_items:
            return (response.output_text or "").strip()
        
        print(f"\n  Round {i + 1}: {len(tool_items)} function(s) called")
        
        # Process each tool call
        for item in tool_items:
            function_name = item.name
            function_args = json.loads(item.arguments)
            
            # Format args for display
            args_str = ", ".join(repr(v) for v in function_args.values())
            print(f"    → {function_name}({args_str})")
            
            # Add function call to conversation
            conversation_items.append({
                "type": "function_call",
                "call_id": item.call_id,
                "name": function_name,
                "arguments": item.arguments
            })
            
            # Execute function and add output
            result = available_functions[function_name](**function_args)
            conversation_items.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result)
            })
    
    return "Error: too many tool-call rounds."


# --------------------------------------------------------------
print("✅ Helper function ready")

### Test the Weather Assistant

In [ ]:
print("🌤️  LIVE WEATHER ASSISTANT")
print("="*60)

answer = run_conversation(
    "What's the weather in Tokyo?",
    weather_tools,
    weather_functions
)
print(answer)

print("="*60)

---

## 💾 Part 2: Database Integration

SQLite for structured storage. We'll build both **read** (SELECT) and **write** (UPDATE) operations.

In [ ]:
# Create in-memory database
def setup_database():
    """Create sample database."""
    conn = sqlite3.connect(':memory:')   # In-memory DB — no file, resets on kernel restart
    cursor = conn.cursor()               # Cursor executes SQL queries
    
    # Create tables
    cursor.execute('''
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            email TEXT,
            plan TEXT
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE orders (
            id INTEGER PRIMARY KEY,
            user_id INTEGER,
            product TEXT,
            amount REAL,
            status TEXT
        )
    ''')
    
    # Insert sample data
    cursor.executemany('INSERT INTO users VALUES (?, ?, ?, ?)', [
        (1, 'Alice', 'alice@example.com', 'premium'),
        (2, 'Bob', 'bob@example.com', 'free'),
        (3, 'Carol', 'carol@example.com', 'premium')
    ])
    
    cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?)', [
        (101, 1, 'Widget', 29.99, 'shipped'),
        (102, 2, 'Gadget', 49.99, 'pending'),
        (103, 1, 'Tool', 19.99, 'delivered')
    ])
    
    conn.commit()
    return conn

# --------------------------------------------------------------
db = setup_database()
print("✅ Database ready")

### Create Database Functions

In [ ]:
# Database functions
def get_user_by_id(user_id):
    """Get user from database."""
    try:
        user_id = int(user_id)
        cursor = db.cursor()
        # IMPORTANT: Use ? placeholder for security (prevents SQL injection)
        cursor.execute('SELECT * FROM users WHERE id = ?', (user_id,))
        row = cursor.fetchone()
        
        if row:
            return {
                "id": row[0],
                "name": row[1],
                "email": row[2],
                "plan": row[3]
            }
        return {"error": "User not found"}
    except Exception as e:
        return {"error": str(e)}

def get_user_orders(user_id):
    """Get all orders for user."""
    try:
        user_id = int(user_id)
        cursor = db.cursor()
        cursor.execute('SELECT * FROM orders WHERE user_id = ?', (user_id,))
        rows = cursor.fetchall()
        
        orders = [
            {
                "order_id": row[0],
                "product": row[2],
                "amount": row[3],
                "status": row[4]
            }
            for row in rows
        ]
        
        return {"orders": orders}
    except Exception as e:
        return {"error": str(e)}

def update_order_status(order_id, new_status):
    """Update order status in database."""
    try:
        order_id = int(order_id)
        cursor = db.cursor()
        cursor.execute('UPDATE orders SET status = ? WHERE id = ?', (new_status, order_id))
        db.commit()
        if cursor.rowcount > 0:
            return {"success": True, "message": f"Order {order_id} updated to '{new_status}'"}
        return {"error": f"Order {order_id} not found"}
    except Exception as e:
        return {"error": str(e)}


# --------------------------------------------------------------
print("✅ Database functions ready")

### Test Database Functions

In [ ]:
# Test
print("Testing database functions...")
print(get_user_by_id(1))
print(get_user_orders(1))
print(update_order_status(102, "shipped"))

### Define Database Tools

In [ ]:
# Database tool schemas (Responses API format)
db_tools = [
    {
        "type": "function",
        "name": "get_user_by_id",
        "description": "Get user information from database by ID",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "integer", "description": "User ID"}
            },
            "required": ["user_id"]
        }
    },
    {
        "type": "function",
        "name": "get_user_orders",
        "description": "Get all orders for a user",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "integer", "description": "User ID"}
            },
            "required": ["user_id"]
        }
    },
    {
        "type": "function",
        "name": "update_order_status",
        "description": "Update the status of an order",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "integer", "description": "Order ID"},
                "new_status": {"type": "string", "description": "New status (e.g. shipped, delivered, refunded)"}
            },
            "required": ["order_id", "new_status"]
        }
    }
]

db_functions = {
    "get_user_by_id": get_user_by_id,
    "get_user_orders": get_user_orders,
    "update_order_status": update_order_status
}


# --------------------------------------------------------------
print("✅ Database tools ready")

### Test Database Assistant

In [ ]:
# Test database assistant
print("💾 DATABASE ASSISTANT")
print("="*60)

answer = run_conversation(
    "Show me all orders for user 1",
    db_tools,
    db_functions
)
print(answer)

print("="*60)

---

## ⚙️ Part 3: Production Integration System

Combine weather APIs and databases into one production-ready system with error handling and logging.

In [ ]:
class IntegrationAssistant:
    """Production assistant with multiple integrations."""
    
    def __init__(self, client, model, instructions=None):
        self.client = client
        self.model = model
        self.instructions = instructions
        self.call_log = []
        self.error_count = 0
        
        # FunctionAssistant Class (M06B) takes tools/functions as parameters (generic).
        # This class bundles them internally (purpose-built for these integrations).
        self.tools = weather_tools + db_tools
        
        # Map function names
        self.functions = {
            "get_live_weather": get_live_weather,
            "get_user_by_id": get_user_by_id,
            "get_user_orders": get_user_orders,
            "update_order_status": update_order_status
        }
    
    def chat(self, user_message, max_turns=5):
        """Chat with comprehensive error handling."""
        # Same loop pattern as FunctionAssistant (M06B), with error handling added
        conversation_items = [{"role": "user", "content": user_message}]
        
        for _ in range(max_turns):
            try:
                response = self.client.responses.create(
                    model=self.model,
                    input=conversation_items,
                    tools=self.tools,
                    instructions=self.instructions
                )
                
                tool_items = [
                    item for item in response.output
                    if item.type == "function_call"
                ]
                
                if not tool_items:
                    return response.output_text
                
                for item in tool_items:
                    function_name = item.name
                    function_args = json.loads(item.arguments)
                    
                    self.call_log.append({
                        "function": function_name,
                        "args": function_args
                    })
                    
                    conversation_items.append({
                        "type": "function_call",
                        "call_id": item.call_id,
                        "name": function_name,
                        "arguments": item.arguments
                    })
                    
                    try:
                        result = self.functions[function_name](**function_args)
                        if isinstance(result, dict) and "error" in result:
                            self.error_count += 1
                    
                    except Exception as e:
                        result = {"error": str(e)}
                        self.error_count += 1
                    
                    conversation_items.append({
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps(result)
                    })
            
            except Exception as e:
                self.error_count += 1
                return f"Error: {str(e)}"
        
        return "Error: too many tool-call rounds."
    
    def stats(self):
        """Get statistics."""
        return {
            "total_calls": len(self.call_log),
            "errors": self.error_count,
            "functions_used": sorted(set(c["function"] for c in self.call_log))
        }


# --------------------------------------------------------------
print("✅ IntegrationAssistant ready")

### Test Production System

In [ ]:
# Test production system
print("⚙️ PRODUCTION INTEGRATION SYSTEM")
print("="*60)

assistant = IntegrationAssistant(
    client=client,
    model=MODEL,
    instructions=INSTRUCTIONS
)

queries = [
    "What's the weather in London?",
    "Show me orders for user 2",
    "Get info for user 3 and check Tokyo weather",
    "Mark order 102 as delivered"
]

for q in queries:
    print(f"\nQ: {q}")
    answer = assistant.chat(q)
    print(f"A: {answer[:400]}..." if len(answer) > 400 else f"A: {answer}")

print("\n" + "="*60)
print("STATISTICS")
stats = assistant.stats()
for key, value in stats.items():
    print(f"{key}: {value}")

---

## 💪 Your Turn: Extend the Agent

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Add Refund Tool
# --------------------------------------------------------------
# Objective: Add a process_refund function that marks an order as "refunded".

def process_refund(order_id):
    """Process refund for an order."""
    # TODO: Convert order_id to int
    # TODO: Check if order exists (SELECT before UPDATE)
    # TODO: Update status to 'refunded'
    # TODO: Return success message or error
    pass

# TODO: Define tool schema
# refund_tool = {
#     "type": "function",
#     "name": "process_refund",
#     ...
# }

# TODO: Create extended assistant
# Hint: Don't modify IntegrationAssistant — create new tool/function lists
# all_tools = assistant.tools + [refund_tool]          # Append new tool
# all_functions = {**assistant.functions, "process_refund": process_refund}  # Merge new function

# TODO: Test with these queries:
# "Refund order 101"
# "Show orders for user 1"  (verify status changed)

# --------------------------------------------------------------
print("💡 Implement the refund tool above!")

---

## 🎯 Key Takeaways

**External API Integration:**
- Wrap API calls in try/except with timeouts
- Return errors as data, not exceptions
- Parse and validate responses before returning

**Database Integration:**
- Use parameterized queries to prevent SQL injection
- Handle both reads (SELECT) and writes (UPDATE) in tools
- Always verify write operations succeeded (check `rowcount`)

**Integration Patterns:**
- Combine multiple tools in a single assistant class
- Log all function calls for debugging
- Track error counts to identify failing integrations
- Extend tool registries with `tools + [new_tool]` and `{**funcs, "name": func}`

---

### 📍 Next Step

Proceed to the **Solutions** notebook to compare your implementation. Congratulations — you've completed the course!

---

## 🔧 Troubleshooting

**API Timeout/Connection Errors?**
- Check internet connection.
- Open-Meteo may be down; try again later.
- Ensure `requests` package is installed (`pip install requests`).

**Database/SQL Errors?**
- Verify database setup (`setup_database()`).
- Ensure parameterized queries use tuples `(value,)`.
- In-memory DB resets if notebook kernel restarts.

**"Expected an input item, but got a string"?**
- When `input` is a list, each item must be a dict
- Use `{"role": "user", "content": message}` not a bare string

**Refund tool not working?**
- Verify order exists before updating (SELECT then UPDATE)
- Check `cursor.rowcount` after UPDATE to confirm change
- Call `db.commit()` to persist the update

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---